# 🌑 LunarSight — Notebook 04: Segmentation Training

**Agent 4**: Weakly-supervised U-Net training using pseudo-labels
from polarimetric diagnostics.

⚡ **GPU Required**: T4 GPU recommended.

---

In [ ]:
# === Setup ===
import os, torch
from google.colab import drive
drive.mount('/content/drive')

REPO_DIR = '/content/Lunar-Sight'
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/YOUR_USERNAME/Lunar-Sight.git {REPO_DIR}
os.chdir(os.path.join(REPO_DIR, 'Lunar-Sight'))
!pip install -q -r requirements_colab.txt

print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"}')

In [ ]:
# === Generate Pseudo-Labels ===
import numpy as np
from agent4_segmentation.pseudo_labels import generate_pseudo_labels

feat = np.load('outputs/agent3/polarimetric_feature_tensor.npy')
l_cpr = feat[0]  # L_CPR channel
dop = feat[2]    # DOP channel (after S_CPR if present)

label_map, stats = generate_pseudo_labels(
    l_cpr=l_cpr, s_cpr=None, dop=dop,
    ice_cpr_min=1.0, ice_dop_max=0.13,
)

os.makedirs('outputs/agent4', exist_ok=True)
np.save('outputs/agent4/pseudo_labels.npy', label_map)
print(f'Pseudo-label stats: {stats}')

In [ ]:
# === Create Dataset ===
import yaml
from torch.utils.data import DataLoader, random_split
from agent4_segmentation.dataset import LunarSegDataset

with open('config/training_config.yaml') as f:
    train_config = yaml.safe_load(f)

seg_cfg = train_config.get('segmentation', {})

dataset = LunarSegDataset(
    feature_tensor_path='outputs/agent3/polarimetric_feature_tensor.npy',
    label_map_path='outputs/agent4/pseudo_labels.npy',
    patch_size=seg_cfg.get('patch_size', 256),
    stride=seg_cfg.get('stride', 128),
)

train_size = int(len(dataset) * 0.8)
val_size = len(dataset) - train_size
train_ds, val_ds = random_split(dataset, [train_size, val_size])

train_loader = DataLoader(train_ds, batch_size=seg_cfg.get('batch_size', 4), shuffle=True)
val_loader = DataLoader(val_ds, batch_size=seg_cfg.get('batch_size', 4))
print(f'Train: {train_size}, Val: {val_size}')

In [ ]:
# === Create Model & Train ===
from agent4_segmentation.model import create_unet
from agent4_segmentation.train import train_segmentation

CHECKPOINT_DIR = '/content/drive/MyDrive/LunarSight/checkpoints/agent4'
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

model = create_unet(in_channels=dataset.n_channels, num_classes=2)

history = train_segmentation(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    config=train_config,
    checkpoint_dir=CHECKPOINT_DIR,
    resume=True,
)

print(f"Best val loss: {history['best_val_loss']:.6f}")

In [ ]:
# === Run Inference ===
from agent4_segmentation.inference import run_segmentation_inference

ice_mask, confidence = run_segmentation_inference(model, feat)

np.save('outputs/agent4/ice_mask.npy', ice_mask)
np.save('outputs/agent4/confidence_map.npy', confidence)

# Visualize
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

axes[0].imshow(label_map, cmap='RdBu', vmin=0, vmax=1)
axes[0].set_title('Pseudo-labels (seed)')

axes[1].imshow(ice_mask, cmap='Blues')
axes[1].set_title(f'Ice Mask ({np.sum(ice_mask)} pixels)')

im = axes[2].imshow(confidence, cmap='viridis', vmin=0, vmax=1)
axes[2].set_title('Confidence Map')
plt.colorbar(im, ax=axes[2])

plt.tight_layout()
plt.show()